# 25. The figures, for a talk

Five figures that tell the unlearning result in order. Each one answers a single question.

**This notebook needs nothing.** No GPU, no credentials, no saved models, no dataset. The
measurements are copied in as plain numbers, so it runs in a few seconds anywhere. Use
Runtime, then Run all.

| Figure | The question it answers |
| --- | --- |
| 1 | What should a chance-level score look like, and what did we see? |
| 2 | Where does the model send the 500 test images? |
| 3 | How much did the weights change, and how much did the behaviour change? |
| 4 | How does the unlearned model differ from one that was never trained? |
| 5 | Can the forgotten task be brought back? |

Section 7 can re-download the saved runs and check that the copied numbers are correct.
That step needs the R2 credentials and is optional. Skipping it changes no figure.

Sources: notebook 21, run `20261012T174517Z`, six completed runs, for figures 1 to 4.
Notebook 24, run `20261014T162221Z`, one seed, for figure 5.

## 1. Setup and the measurements

In [ ]:
# Only matplotlib and numpy. No GPU, no R2 credentials, no checkpoints.
# Every number below is copied from a saved run; section 7 can re-download
# them and check that these copies are correct.
import numpy as np
import matplotlib.pyplot as plt
from math import comb

plt.rcParams.update({
    "figure.dpi": 120, "savefig.dpi": 200, "savefig.bbox": "tight",
    "font.size": 11, "axes.titlesize": 12, "axes.titleweight": "normal",
    "axes.spines.top": False, "axes.spines.right": False,
    "axes.grid": True, "grid.alpha": 0.25, "grid.linewidth": 0.6,
    "legend.frameon": False,
})
BLUE, ORANGE, AQUA, GREY = "#2a78d6", "#eb6834", "#1baf7a", "#888780"

# ---- the saved measurements ------------------------------------------------
# Six completed runs, notebook 21, R2 diagnostics/20261012T174517Z
RUNS = [  # seed, forgotten task, accuracy before, accuracy after, sensitivity after
    (0, "3",  35.6, 10.0, 2.28e-06),
    (0, "14", 42.6, 10.0, 2.68e-06),
    (1, "3",  40.4, 10.0, 8.94e-08),
    (1, "14", 40.4, 10.0, 2.98e-07),
    (2, "3",  38.4, 10.0, 7.45e-08),
    (2, "14", 41.8, 10.0, 1.49e-07),
]
CLASSES, VAL_IMAGES = 10, 500

# Seed 0 forgetting task 3, measured every 20 unlearning steps
STEPS       = [0, 20, 40, 60, 80, 100]
WEIGHT_SIZE = [4706.2, 4007.7, 3542.3, 3191.9, 2921.1, 2690.3]
SENSITIVITY = [1.324e1, 3.94e-1, 2.59e-2, 1.74e-3, 7.39e-5, 2.28e-6]

# How many of the 500 test images went to each class, same run
COUNTS = {
    0:   [86, 113, 21, 3, 65, 0, 108, 1, 89, 14],
    20:  [0, 0, 0, 494, 6, 0, 0, 0, 0, 0],
    40:  [0, 0, 0, 500, 0, 0, 0, 0, 0, 0],
    100: [500, 0, 0, 0, 0, 0, 0, 0, 0, 0],
}

# A never-trained network, measured on three separate builds
NEVER_TRAINED = [38.2, 95.0, 309.4]

# Recovery with the generator frozen, R2 diagnostics/20261014T162221Z, one seed.
# Best accuracy reached in 300 steps.
BUDGET_IMAGES = [10, 50, 100, 200]
REC_OLD_CODE  = [10.4, 10.2, 10.0, 10.0]
REC_FRESH     = [12.4, 21.0, 15.2, 11.0]
REC_UNSEEN    = [12.0, 10.6, 10.9, 10.9]
ORIGINAL_ACC  = 35.6

CHANCE = 100.0 / CLASSES
print(f"{len(RUNS)} runs loaded. Chance level is {CHANCE:.0f}%.")
print("Every run ended at exactly 10.0%:", all(r[3] == 10.0 for r in RUNS))

## 2. What a chance-level score should look like

Each task is scored on 500 images, 50 from each of 10 classes. If the model were really
guessing, its score would move around. This figure shows how much.

**What to say:** a guessing model lands anywhere in the blue region, and the number changes
each run. Ours was exactly 10.0% six times out of six. That is not guessing.

In [ ]:
# Figure 1. What a chance-level score should look like, and what we saw.
k = np.arange(30, 71)
prob = np.array([comb(VAL_IMAGES, int(i)) * 0.1**i * 0.9**(VAL_IMAGES - i) for i in k])
acc = 100.0 * k / VAL_IMAGES

fig, ax = plt.subplots(figsize=(8, 4.1))
ax.bar(acc, prob, width=0.17, color=GREY, label="where a guessing model would land")
inside = (acc >= 7.3) & (acc <= 12.7)
ax.bar(acc[inside], prob[inside], width=0.17, color=BLUE,
       label="usual range for guessing, 7.3% to 12.7%")
ax.axvline(10.0, color=ORANGE, lw=2.5)
ax.annotate("all six runs landed\nexactly here", xy=(10.0, prob.max() * 0.88),
            xytext=(12.1, prob.max() * 0.92), color=ORANGE, fontsize=10,
            arrowprops=dict(arrowstyle="->", color=ORANGE, lw=1.4))
ax.set_xlabel("accuracy on the forgotten task (%)")
ax.set_ylabel("probability")
ax.set_title("A guessing model would scatter. Ours did not move.")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.24), ncol=2, fontsize=9)
plt.show()

print(f"Chance of landing on exactly 10.0% once : {prob[k == 50][0]:.3f}")
print(f"Chance of doing that in all six runs    : {prob[k == 50][0] ** 6:.2e}")

## 3. Where the model sends the images

The same run, counted by class. This is the direct explanation of the 10.0%.

**What to say:** at the start the answers spread across nine classes. By step 40 all 500
images get the same answer. One fixed answer is correct for the 50 images of that class, so
the score is exactly 50 out of 500.

In [ ]:
# Figure 2. Where the model sends the 500 test images, during unlearning.
fig, axes = plt.subplots(1, 4, figsize=(11, 2.9), sharey=True)
for ax, step in zip(axes, [0, 20, 40, 100]):
    counts = COUNTS[step]
    used = sum(1 for c in counts if c > 0)
    ax.bar(range(CLASSES), counts, color=BLUE if used > 1 else ORANGE, width=0.72)
    ax.set_title(f"step {step}\n{used} of 10 classes used", fontsize=10)
    ax.set_xticks(range(CLASSES))
    ax.set_xlabel("predicted class")
axes[0].set_ylabel("images (of 500)")
fig.suptitle("By step 40 every image gets the same answer", y=1.06)
plt.show()

print("50 images per class, so one fixed answer is right 50 of 500, which is exactly 10.0%.")

## 4. Small change in weights, total change in behaviour

Both lines start at 1 and are shown as a fraction of their starting value, on a
logarithmic scale.

**What to say:** the weights only lose 43% of their size. The model's response to the image
falls by about ten million times. A deep network multiplies its signal through roughly 50
layers, so small reductions compound.

In [ ]:
# Figure 3. The weights fall a little. The response to the image collapses.
w = np.array(WEIGHT_SIZE) / WEIGHT_SIZE[0]
s = np.array(SENSITIVITY) / SENSITIVITY[0]

fig, ax = plt.subplots(figsize=(8, 4))
ax.plot(STEPS, w, "-o", color=BLUE, lw=2.5, ms=6, label="size of the generated weights")
ax.plot(STEPS, s, "--s", color=ORANGE, lw=2.5, ms=6, label="input sensitivity")
ax.set_yscale("log")
ax.set_ylim(1e-8, 3)
ax.set_xlabel("unlearning step")
ax.set_ylabel("fraction of the value at step 0")
ax.set_title("A 43% cut in the weights, a ten-million-fold cut in the response")
ax.legend(loc="lower left")
ax.annotate(f"ends at {w[-1]:.2f}", xy=(100, w[-1]), xytext=(74, 0.18), color=BLUE,
            arrowprops=dict(arrowstyle="->", color=BLUE, lw=1.2))
ax.annotate(f"ends at {s[-1]:.1e}", xy=(100, s[-1]), xytext=(60, 2e-7), color=ORANGE,
            arrowprops=dict(arrowstyle="->", color=ORANGE, lw=1.2))
plt.show()

print("Why: a deep network multiplies its signal through about 50 layers.")
print(f"If each layer keeps 72% of its size:  0.72 ** 50 = {0.72 ** 50:.1e}")

## 5. Three states that all score about 10%

Input sensitivity is how much the model's ten output scores move between different
pictures. Near zero means the model is not reading the picture.

**What to say:** the paper says unlearning leaves the task like a network that was never
trained. A never-trained network still reacts to the image. This one does not.

In [ ]:
# Figure 4. Three states that all score about 10%, and how they differ.
labels = ["working model\n(before unlearning)",
          "never trained\n(the paper's claim)",
          "after unlearning\n(what we measured)"]
low  = [min(10.8, 13.9), min(NEVER_TRAINED), min(r[4] for r in RUNS)]
high = [13.9, max(NEVER_TRAINED), max(r[4] for r in RUNS)]
colours = [BLUE, AQUA, ORANGE]

fig, ax = plt.subplots(figsize=(8, 3.2))
for i, (lo, hi, c) in enumerate(zip(low, high, colours)):
    ax.plot([lo, hi], [i, i], color=c, lw=9, solid_capstyle="butt")
    ax.plot([lo, hi], [i, i], "o", color=c, ms=7)
    ax.text(hi * 1.7, i, f"{lo:.3g} to {hi:.3g}", va="center", fontsize=10, color=c)
ax.set_xscale("log")
ax.set_xlim(1e-8, 3e4)
ax.set_yticks(range(3)); ax.set_yticklabels(labels, fontsize=10)
ax.invert_yaxis()
ax.set_xlabel("input sensitivity (log scale)")
ax.set_title("All three score near 10%. Only one has stopped reading the image.")
ax.grid(axis="y", visible=False)
plt.show()

print("A never-trained network knows nothing but still reacts to the picture.")
print("The unlearned model reacts about a hundred million times less.")

## 6. Can the task be brought back?

This uses the authors' own procedure: freeze the generator, train only a fresh task code on
a small sample of the forgotten task's images.

**What to say:** nothing approaches the original 35.6%. More images do not help. And tasks
the model has never seen do just as well, so no knowledge of the forgotten task is left
behind. One seed, so this is provisional.

In [ ]:
# Figure 5. Recovery with the generator frozen, which is the authors' own test.
x = np.arange(len(BUDGET_IMAGES)); w = 0.26
fig, ax = plt.subplots(figsize=(8, 4.5))
ax.bar(x - w, REC_OLD_CODE, w, color=BLUE,   label="the forgotten task's own code")
ax.bar(x,     REC_FRESH,    w, color=ORANGE, label="a freshly created code")
ax.bar(x + w, REC_UNSEEN,   w, color=AQUA,   label="tasks the model never saw")
ax.axhline(ORIGINAL_ACC, color="black", ls="--", lw=1.2)
ax.text(len(x) - 0.55, ORIGINAL_ACC + 0.7, f"before unlearning, {ORIGINAL_ACC}%", fontsize=9)
ax.axhline(CHANCE, color=GREY, ls=":", lw=1.2)
ax.text(len(x) - 0.55, CHANCE + 0.7, f"chance, {CHANCE:.0f}%", fontsize=9, color=GREY)
ax.set_xticks(x); ax.set_xticklabels([f"{n} images" for n in BUDGET_IMAGES])
ax.set_ylim(0, 40)
ax.set_ylabel("best accuracy reached (%)")
ax.set_title("Nothing comes back, and more data does not help")
ax.legend(loc="upper center", bbox_to_anchor=(0.5, -0.14), ncol=3, fontsize=9)
plt.show()

print("The forgotten task's own code never moves above 10.4%.")
print("Tasks the model has never seen do just as well, so nothing is hiding inside.")

## 7. Optional: check the numbers against the saved runs

Downloads the saved result files and compares them with the numbers used above. Needs
`R2_ENDPOINT`, `R2_ACCESS_KEY_ID` and `R2_SECRET_ACCESS_KEY`. Without them the cell reports
that it was skipped and nothing else changes.

In [ ]:
# Optional. Re-download the saved runs and check the numbers above.
# Needs the three R2 secrets. Skip this cell for a demo; it changes no figure.
import os, json, glob, tempfile

def secret(name):
    value = os.environ.get(name)
    if not value:
        try:
            from google.colab import userdata
            value = userdata.get(name)
        except Exception:
            value = None
    return value

creds = [secret(n) for n in ("R2_ENDPOINT", "R2_ACCESS_KEY_ID", "R2_SECRET_ACCESS_KEY")]
if not all(creds):
    print("No R2 credentials found, so the check was skipped.")
    print("The figures above are unaffected.")
else:
    import boto3
    endpoint, key, secret_key = creds
    s3 = boto3.client("s3", endpoint_url=endpoint,
                      aws_access_key_id=key, aws_secret_access_key=secret_key)
    prefix = "uncle/paired_generalization/paired_generalization_v2/diagnostics/20261012T174517Z/"
    found = s3.list_objects_v2(Bucket="uncle-experiments", Prefix=prefix).get("Contents", [])
    path = next(o["Key"] for o in found if o["Key"].endswith("summary.json"))
    local = tempfile.mktemp(suffix=".json")
    s3.download_file("uncle-experiments", path, local)
    saved = json.load(open(local))

    ok = True
    for row, copied in zip(saved["comparison"], RUNS):
        live = (row["seed"], row["forget"], row["start_accuracy"],
                row["end_accuracy"], row["sensitivity_after"])
        same = (live[0] == copied[0] and live[1] == copied[1]
                and abs(live[2] - copied[2]) < 0.05 and abs(live[3] - copied[3]) < 1e-9
                and abs(live[4] - copied[4]) / copied[4] < 0.01)
        ok = ok and same
        print(f"  seed {live[0]} forget {live[1]:<2} : {'matches' if same else 'DIFFERS'}")

    rows = saved["runs"][0]["rows"]
    w_same = all(abs(r["weight_size"] - c) < 0.1 for r, c in zip(rows, WEIGHT_SIZE))
    c_same = all(rows[i]["class_counts"] == COUNTS[rows[i]["step"]]
                 for i in range(len(rows)) if rows[i]["step"] in COUNTS)
    print(f"  weight trajectory  : {'matches' if w_same else 'DIFFERS'}")
    print(f"  prediction counts  : {'matches' if c_same else 'DIFFERS'}")
    print()
    print("ALL NUMBERS VERIFIED" if (ok and w_same and c_same) else "A NUMBER DIFFERS")

## 8. Save the figures

In [ ]:
# Save every figure as PNG, for slides or a document.
import os
os.makedirs("figures", exist_ok=True)
for i, fig_num in enumerate(plt.get_fignums(), start=1):
    plt.figure(fig_num).savefig(f"figures/figure_{i}.png")
print("Written:", sorted(os.listdir("figures")))
print("In Colab, download them from the file browser on the left.")

## Notes for the talk

**Figures 1 to 4 rest on six runs.** Figure 5 rests on one seed and should be presented as
provisional.

**Two things not to claim.** There is no evidence that the forgotten information leaks;
figure 5's third bar is what rules it out. And this is not the paper's configuration, so
it does not reproduce their reported numbers.

**The one sentence the figures build to:** the method reports a task as forgotten once its
accuracy reaches chance, and it reaches chance by making the model stop responding to its
input rather than by returning it to an untrained state.